# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing a **freestyle lane focused on content performance intelligence and action prioritization**. The project will investigate whether observable search, traffic, engagement, freshness and performance signals can help identify meaningful content-performance states and prioritize **which page should be reviewed next, why it deserves attention, and what type of action a reviewer should consider**. Signal analysis, clustering, CTR or engagement gap analysis, and ranking or scoring may be used where the evidence supports them, rather than being forced into the project. The final output should remain human-reviewed and explainable, with a priority score, performance context, reason codes, confidence and a suggested action such as refresh, improve, protect, monitor or deprioritize.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    data_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

df.head()


## 2. The question: decision, action, cost of a wrong call

### Research question

Can observable search, traffic, engagement, freshness and performance signals identify meaningful content opportunities or risks and help determine **which pages should be reviewed first, why they deserve attention, and what type of action should be considered?**

### Unit of analysis

The main unit of analysis is one pseudonymized content page at a defined decision point, using its historical performance up to that point.

### Decision

The decision is how a content, SEO or marketing team should allocate limited human review time across a large inventory of existing pages.

### Output

The intended output is a ranked human-review queue. Each recommended page could eventually include:

- a priority score;
- a performance state or context;
- reason codes explaining why it was ranked highly;
- a confidence level;
- and a suggested action such as refresh, expand, improve CTR, protect, monitor or deprioritize.

### Action

A human reviewer would start with the highest-priority pages, inspect the recommendation and its supporting evidence, and then decide whether any content or SEO intervention is appropriate.

The system would support the decision rather than automatically editing or publishing content.

### Cost of a wrong recommendation

A false positive could waste editorial or marketing time and could encourage an unnecessary change to a healthy page.

A false negative could cause a meaningful decline or opportunity to be missed.

A misleading suggested action could also direct effort toward the wrong problem. For example, a page with low CTR may not actually need a title change if its position, search intent or volume explains the observed performance.

Because the cost is mainly wasted human effort and missed opportunity, the quality and explainability of the highest-ranked recommendations are especially important.

### Why data or ML can help

The content inventory contains many pages and multiple interacting signals, making manual prioritization difficult at scale.

Data analysis can first establish which signals and patterns are useful. More advanced methods such as clustering or predictive models may then help discover structure or combine evidence that would be difficult to capture with a single fixed rule.

However, ML is not automatically better. A transparent rule-based baseline will be built first, and more complex methods should only be retained if they provide a useful and validated improvement.

In [ ]:
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "trend_direction"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing required columns:", missing_columns)

print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Total rows: {len(df):,}")

if df["content_id"].nunique() == len(df):
    print("Unit-of-analysis check passed: one row per content page.")
else:
    print("Warning: duplicate content IDs found.")


## 3. Quick look at the data (2-3 real numbers)

Three checks from the starter dataset suggest that simple trend classification alone would not be enough for useful content prioritization.

Among pages with at least **3,000 impressions over 90 days**, **57.0% are in the `down` trend category**. This means that even among pages with substantial observed visibility, a large group may require further prioritization rather than treating all declining pages equally.

Among pages averaging a **page-one position** and receiving at least **300 impressions**, **10.2% received zero clicks** during the 90-day window. This suggests that some visible pages may represent a distinct CTR or search-result opportunity rather than simply a general performance decline.

Finally, **43.0% of high-visibility declining pages had not been updated for at least 90 days**. Freshness therefore appears potentially relevant for part of the declining population, while also showing that staleness alone cannot explain every decline.

Together, these observations support a broader decision problem: identifying different kinds of content opportunity or risk, understanding why they occur, and prioritizing which pages deserve human review first. These are descriptive observations from the starter data and do not establish that any particular intervention will cause performance to improve.

In [ ]:
high_visibility = df["impressions_90d"] >= 3000
moderate_visibility = df["impressions_90d"] >= 300

down = df["trend_direction"].eq("down")

page_one = (
    df["avg_position"].gt(0)
    & df["avg_position"].le(10)
)

zero_clicks = df["clicks_90d"].eq(0)
stale = df["days_since_last_update"].ge(90)

high_visibility_down = high_visibility & down
page_one_visible = page_one & moderate_visibility

high_visibility_down_pct = (
    100 * high_visibility_down.sum()
    / high_visibility.sum()
)

page_one_zero_click_pct = (
    100 * (page_one_visible & zero_clicks).sum()
    / page_one_visible.sum()
)

stale_high_visibility_down_pct = (
    100 * (high_visibility_down & stale).sum()
    / high_visibility_down.sum()
)

print(
    "High-visibility pages in down trend: "
    f"{high_visibility_down.sum():,} / {high_visibility.sum():,} "
    f"({high_visibility_down_pct:.1f}%)"
)

print(
    "Page-one visible pages with zero clicks: "
    f"{(page_one_visible & zero_clicks).sum():,} / "
    f"{page_one_visible.sum():,} "
    f"({page_one_zero_click_pct:.1f}%)"
)

print(
    "High-visibility down pages stale >= 90 days: "
    f"{(high_visibility_down & stale).sum():,} / "
    f"{high_visibility_down.sum():,} "
    f"({stale_high_visibility_down_pct:.1f}%)"
)


## 4. Careful words: what I can and can't claim

### What this project may be able to say

The project can identify **observed relationships and recurring patterns** in the available search, traffic, engagement, freshness and content-performance data.

It may identify groups of pages with similar measurable characteristics and determine whether particular combinations of signals are useful for prioritizing human review.

It may also compare a transparent baseline with more advanced analytical or ML methods and report whether the latter improve the chosen ranking or validation metrics.

The resulting recommendations should be described as **directional, evidence-based decision support**.

### What this project cannot claim

The project cannot prove that a suggested intervention will cause search traffic or rankings to improve.

The project cannot infer how Google's ranking algorithm works.

A page showing declining traffic, weak CTR or another apparent problem does not automatically require a particular intervention.

Clusters or performance states, if used, are analytical groupings rather than objectively true content categories.

The current starter trend fields also describe an existing measurement window rather than an ideal future outcome. A stronger later version of the project should therefore use clearly separated time windows where possible:

**past feature window → decision point → future outcome window**

This would allow later modeling to test whether current observable signals help predict a genuinely future decline, recovery or opportunity without leaking information from the outcome period.

In [ ]:
product_decision_fields = {
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
}

present_product_fields = sorted(
    product_decision_fields.intersection(df.columns)
)

print("Product decision fields present:", present_product_fields)

assert not present_product_fields, (
    "Product decision fields found — review for circularity."
)

print()
print("Leakage reminder:")
print(
    "`trend_direction` is derived from `trend_pct`, "
    "so neither should later be used as a model feature "
    "if the target is based on the same trend definition."
)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.